# Pydantic格式的使用

## 基本使用

举例1：

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名30岁的软件工程师")
rprint(res)
rprint(type(res))

In [ ]:
rprint(f"姓名:{res.name}")
rprint(f"年龄:{res.age}")
rprint(f"职业:{res.occupation}")

举例2：

In [ ]:
class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

structured_model = model.with_structured_output(MovieModel)


res = structured_model.invoke("给出电影盗梦空间的信息")
rprint(res)

举例3：

In [ ]:
from pydantic import BaseModel, Field

class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度：0-1之间")
    keywords: list[str] = Field(description="关键词列表")

structured_model = model.with_structured_output(SentimentAnalysis)

text = "这个课程很有用，学到了很多东西，强烈推荐"
res = structured_model.invoke(
    f"分析一下文本的情感：\n{text}"
)

rprint(f"类型:{type(res)}")
rprint(f"情感:{res.sentiment}")
rprint(f"置信度:{res.confidence}")
rprint(f"关键词:{res.keywords}")

## 2.高级特性

### 2.1 情况1：可选字段

举例：

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

作为对比：

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Optional

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

### 2.2 情况2：默认值

不同的模型供应商对于此字段的支持是不同的。

举例1：使用deepseek模型

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)

class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(default = 10, description="年龄")  # 默认值default = 10
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

In [ ]:
class Person(BaseModel):
    """人物的信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄(默认值为20)")  # 相当于提示词
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

res = structured_model.invoke("张三是一名软件工程师")
rprint(res)
rprint(type(res))

### 2.3 情况3：枚举类型

方式一：

In [ ]:
from typing import Optional
from pydantic import Field
from enum import Enum

# 定义枚举类型
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_model = model.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

print("\n提取结果：")
print(f"  客户：{result.name}")
print(f"  电话：{result.phone}")
print(f"  邮箱：{result.email or '未提供'}")
print(f"  问题：{result.issue}")
print(f"  紧急程度：{result.urgency.value}")

方式二：

In [ ]:
from typing import Optional, Literal
from pydantic import Field

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")

# 测试
structured_model = model.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_model.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")

print(result)

print("\n提取结果：")
print(f"  客户：{result.name}")
print(f"  电话：{result.phone}")
print(f"  邮箱：{result.email or '未提供'}")
print(f"  问题：{result.issue}")

### 2.4 情况4：列表提取

举例1：

In [ ]:
from typing import List
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)


class Preson(BaseModel):
    """人物信息"""
    name : str = Field(description="姓名")
    age : int = Field(description="年龄")

class PersonList(BaseModel):
    """任务列表"""
    people : List[Person]   # 多个Person的对象

structured_model = model.with_structured_output(PersonList)

res = structured_model.invoke("张三30岁，李斯20岁")
print(res)

举例2：

In [ ]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")

print(review)

举例3：

In [ ]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")

# 测试
structured_llm = model.with_structured_output(Invoice)

invoice_text = """
发票号：INV-2024-001
日期：2024-01-15
总金额：1299.00
商品：MacBook Pro, AppleCare+
"""

invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

### 2.5 情况5:嵌套结构

举例1：

In [ ]:
class Address(BaseModel):
    """地点描述"""
    city : str = Field(description="城市")
    district : str = Field(description="区域")

class Company(BaseModel):
    name : str = Field(description="公司名称")
    address : Address = Field(description="公司所在地")

structured_model = model.with_structured_output(Company)

res = structured_model.invoke("阿里巴巴在杭州的滨江区")
print(res)

举例2：

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")

# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")

# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

举例3：

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Aspect(BaseModel):
    """评论维度"""
    name : str = Field(description="维度名称，如质量、价格、服务")
    score : str = Field(description="评分， 1-5")
    comment : str = Field(description="具体评价")

class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感: positive/negative/neutral")
    overall_score: int = Field(description="综合评分, 1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")

# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)

# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""

result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)

print(f"整体情感: {result.overall_sentiment}")
print(f"综合评分: {result.overall_score}/5")
print(f"\n各维度评价:")
for aspect in result.aspects:
    print(f"  - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结: {result.summary}")

## 2.6 限制条件